# Day 4 - 필수 과제 1: PDF 가공과 LangChain 검색

이 노트북은 Notion의 `[필수] 과제1` 요구사항을 기준으로 작성했습니다.

- `가이드라인_발췌.pdf`: 서술형 PDF를 로드하고 정리한 뒤 markdown 구조로 변환
- `가이드라인_표포함.pdf`: 표가 있는 PDF를 `PyPDFLoader`와 `pdfplumber`로 비교
- 마지막으로 정리된 markdown을 LangChain에 넣어 검색/질의응답까지 연결


## 0. 실행 환경

이 프로젝트의 `.venv`에 필요한 패키지가 이미 설치되어 있다고 가정합니다. OpenAI 기반 답변까지 실행하려면 프로젝트 루트의 `.env`에 `OPENAI_API_KEY`가 있어야 합니다.

패키지 설치 셀은 제거했습니다. 현재 프로젝트 환경에서는 별도 설치 없이 바로 실행합니다.

In [6]:
import os
import re
from pathlib import Path

import pandas as pd
import pdfplumber
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_core.documents import Document

CANDIDATE_DIRS = [
    Path('.').resolve(),
    (Path('.').resolve() / 'HW' / 'NLP' / 'day4').resolve(),
]

BASE_DIR = next((p for p in CANDIDATE_DIRS if (p / '가이드라인_발췌.pdf').exists()), CANDIDATE_DIRS[0])
TEXT_PDF = BASE_DIR / '가이드라인_발췌.pdf'
TABLE_PDF = BASE_DIR / '가이드라인_표포함.pdf'
FULL_PDF = BASE_DIR / '「보건의료데이터+활용+가이드라인」(2025.1.).pdf'

ROOT_ENV = next((parent / '.env' for parent in [BASE_DIR, *BASE_DIR.parents] if (parent / '.env').exists()), BASE_DIR / '.env')

load_dotenv(ROOT_ENV)

def build_fallback_pypdf_loader():
    class FallbackPyPDFLoader:
        def __init__(self, file_path: str):
            self.file_path = file_path

        def load(self):
            reader = PdfReader(self.file_path)
            docs = []
            for i, page in enumerate(reader.pages):
                text = page.extract_text() or ''
                docs.append(Document(page_content=text, metadata={'source': self.file_path, 'page': i}))
            return docs

    return FallbackPyPDFLoader

PyPDFLoader = build_fallback_pypdf_loader()
PDF_LOADER_BACKEND = 'fallback_pypdf'

for path in [TEXT_PDF, TABLE_PDF, FULL_PDF]:
    print(path.name, 'exists ->', path.exists())
print('PDF loader backend ->', PDF_LOADER_BACKEND)

가이드라인_발췌.pdf exists -> True
가이드라인_표포함.pdf exists -> True
「보건의료데이터+활용+가이드라인」(2025.1.).pdf exists -> True
PDF loader backend -> fallback_pypdf


## 1단계 - 외부 데이터 불러오기

`PyPDFLoader`가 PDF를 페이지 단위 `Document`로 바꾸는지 확인합니다. 이 단계에서 과제 요구대로 `page_content`와 `metadata`를 직접 확인합니다.

In [7]:
loader = PyPDFLoader(str(TEXT_PDF))
docs = loader.load()

print('문서 개수:', len(docs))
print('첫 번째 metadata:', docs[0].metadata)
print('\n첫 번째 page_content 미리보기:\n')
print(docs[0].page_content[:1200])

문서 개수: 5
첫 번째 metadata: {'source': '/mnt/c/Users/sdh08/PycharmProjects/PythonProject1/HW/NLP/day4/가이드라인_발췌.pdf', 'page': 0}

첫 번째 page_content 미리보기:

02  보건의료데이터 활용 가이드라인
1 필요성 및 목적
개정된 개인정보 보호법(이하 ‘보호법’ 이라 함)이 시행(’20.8.5)됨에 따라, 데이터 활용의 핵심인 
가명정보 활용에 대한 법적 근거 마련(제3절 특례조항 신설)
- 개인정보처리자가 개인정보를 가명처리하여 통계작성, 과학적 연구, 공익적 기록보존 등의 목적으로 
활용할 수 있는 기반 마련
개인정보 보호 법령 등에서 구체적으로 정하지 않은 가명처리, 가명정보의 처리 및 결합 활용 
등에 있어 보건의료데이터의 특수성 고려 필요
- 보건의료데이터의 분야 ･유형･목적별 세부 방법과 절차를 제시하여 가명정보의 처리에 대한 이해를 
돕고, 자료 오남용 방지
- 처리 과정 전반에 걸쳐 절차 및 거버넌스, 안전조치, 윤리적 사항 등을 정하여 정보 주체의 권익을 
보호하고 안전한 개인정보 처리 도모
2 관련 근거
관련 법령
- ｢개인정보 보호법 ｣ 제2조(정의), 제3절 가명정보의 처리에 관한 특례
- ｢개인정보 보호법 시행령 ｣ 제4장의2 가명정보의 처리에 관한 특례
고시 등
- ｢가명정보의 결합 및 반출 등에 관한 고시 ｣
- ｢가명정보 처리 가이드라인 ｣ 
▷ 개인정보처리자가 법에 따른 규정을 준수한 경우 가이드라인 미준수를 사유로 처벌받지 않음, 따라서, 개인정보
처리자는 데이터의 관련 분야 및 특수성 등을 고려하여 상황에 따라 유동적으로 처리 가능함
    ※ 가명정보 처리 가이드라인(개인정보보호위원회(’24.2.))
01 가이드라인 개요


## 2단계 - 텍스트로 정리하기

목표는 `사람이 읽기에도 좋고, 모델이 처리하기에도 좋은 clean text`를 만드는 것입니다.

- 페이지별 텍스트를 하나로 합칩니다.
- 머리말, 페이지 번호, 과도한 공백, 불필요한 줄바꿈을 줄입니다.
- 정리 전/후를 비교해 어떤 노이즈가 제거됐는지 확인합니다.


In [8]:
raw_text = '\n\n'.join(doc.page_content for doc in docs)

def clean_text(text: str) -> str:
    text = text.replace('\u200b', ' ')
    text = text.replace('\xa0', ' ')
    text = re.sub(r'보건의료데이터 활용 가이드라인', ' ', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    text = re.sub(r'(?<=[가-힣A-Za-z0-9,])\n(?=[가-힣A-Za-z0-9])', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()

cleaned_text = clean_text(raw_text)

print('[정리 전 일부]\n')
print(raw_text[:1500])
print('\n' + '=' * 100 + '\n')
print('[정리 후 일부]\n')
print(cleaned_text[:1500])

[정리 전 일부]

02  보건의료데이터 활용 가이드라인
1 필요성 및 목적
개정된 개인정보 보호법(이하 ‘보호법’ 이라 함)이 시행(’20.8.5)됨에 따라, 데이터 활용의 핵심인 
가명정보 활용에 대한 법적 근거 마련(제3절 특례조항 신설)
- 개인정보처리자가 개인정보를 가명처리하여 통계작성, 과학적 연구, 공익적 기록보존 등의 목적으로 
활용할 수 있는 기반 마련
개인정보 보호 법령 등에서 구체적으로 정하지 않은 가명처리, 가명정보의 처리 및 결합 활용 
등에 있어 보건의료데이터의 특수성 고려 필요
- 보건의료데이터의 분야 ･유형･목적별 세부 방법과 절차를 제시하여 가명정보의 처리에 대한 이해를 
돕고, 자료 오남용 방지
- 처리 과정 전반에 걸쳐 절차 및 거버넌스, 안전조치, 윤리적 사항 등을 정하여 정보 주체의 권익을 
보호하고 안전한 개인정보 처리 도모
2 관련 근거
관련 법령
- ｢개인정보 보호법 ｣ 제2조(정의), 제3절 가명정보의 처리에 관한 특례
- ｢개인정보 보호법 시행령 ｣ 제4장의2 가명정보의 처리에 관한 특례
고시 등
- ｢가명정보의 결합 및 반출 등에 관한 고시 ｣
- ｢가명정보 처리 가이드라인 ｣ 
▷ 개인정보처리자가 법에 따른 규정을 준수한 경우 가이드라인 미준수를 사유로 처벌받지 않음, 따라서, 개인정보
처리자는 데이터의 관련 분야 및 특수성 등을 고려하여 상황에 따라 유동적으로 처리 가능함
    ※ 가명정보 처리 가이드라인(개인정보보호위원회(’24.2.))
01 가이드라인 개요

제1장 | 가이드라인 개요 03
3 적용 범위
(우선순위) 보건의료 분야의 개인정보 가명처리 및 가명정보 처리에 관하여 동 가이드라인 직용
※ 본 가이드라인에서 별도로 정하지 않은 개인정보 보호 법령 및 고시에 규정된 사항은 ｢가명정보 처리 가이드라인｣ 
(개인정보보호위원회)을 준용해야 함
  - 또한 ｢가명정보 처리 가이드라인｣ (개인정보보호위원회)에서 구체적으로 제시하고 있는 가명처리 및 가명정보 처리에 
관한 예시나 서식 등에 대해서는 개인정보

## 3단계 - markdown 구조로 변환하기

과제의 핵심 단계입니다. 평평한 텍스트를 그대로 두지 않고, 제목/소제목/항목 구조를 살린 markdown으로 바꿉니다.

아래 함수는 숫자 제목과 용어 항목을 기준으로 기본적인 구조를 입히도록 작성했습니다. 실제 문서의 줄바꿈 상태에 따라 세부 규칙은 조금 조정될 수 있습니다.

In [9]:
def text_to_markdown(text: str) -> str:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    md_lines = ['# 보건의료데이터 활용 가이드라인 개요']

    for line in lines:
        if line == '보건의료데이터 활용 가이드라인 개요':
            continue
        if re.match(r'^제\d+장', line):
            md_lines.append(f'## {line}')
        elif re.match(r'^\d+\.\s*', line):
            md_lines.append(f'## {line}')
        elif re.match(r'^[①-⑳]', line):
            md_lines.append(f'- {line}')
        elif ':' in line and len(line) < 80:
            md_lines.append(f'- **{line}**')
        elif any(keyword in line for keyword in ['가명처리', '정보주체', '결합키', '개인정보처리자']):
            md_lines.append(f'- {line}')
        else:
            if md_lines and not md_lines[-1].startswith(('#', '- ')):
                md_lines[-1] += ' ' + line
            else:
                md_lines.append(line)

    markdown_text = '\n\n'.join(md_lines)
    markdown_text = re.sub(r'\n{3,}', '\n\n', markdown_text)
    return markdown_text.strip()

markdown_text = text_to_markdown(cleaned_text)
print(markdown_text[:2500])

# 보건의료데이터 활용 가이드라인 개요

02 1 필요성 및 목적 개정된 개인정보 보호법(이하 ‘보호법’ 이라 함)이 시행(’20.8.5)됨에 따라, 데이터 활용의 핵심인 가명정보 활용에 대한 법적 근거 마련(제3절 특례조항 신설)

- - 개인정보처리자가 개인정보를 가명처리하여 통계작성, 과학적 연구, 공익적 기록보존 등의 목적으로

- 활용할 수 있는 기반 마련 개인정보 보호 법령 등에서 구체적으로 정하지 않은 가명처리, 가명정보의 처리 및 결합 활용

등에 있어 보건의료데이터의 특수성 고려 필요 - 보건의료데이터의 분야 ･유형･목적별 세부 방법과 절차를 제시하여 가명정보의 처리에 대한 이해를 돕고, 자료 오남용 방지 - 처리 과정 전반에 걸쳐 절차 및 거버넌스, 안전조치, 윤리적 사항 등을 정하여 정보 주체의 권익을 보호하고 안전한 개인정보 처리 도모 2 관련 근거 관련 법령 - ｢개인정보 보호법 ｣ 제2조(정의), 제3절 가명정보의 처리에 관한 특례 - ｢개인정보 보호법 시행령 ｣ 제4장의2 가명정보의 처리에 관한 특례 고시 등 - ｢가명정보의 결합 및 반출 등에 관한 고시 ｣ - ｢가명정보 처리 가이드라인 ｣

- ▷ 개인정보처리자가 법에 따른 규정을 준수한 경우 가이드라인 미준수를 사유로 처벌받지 않음, 따라서, 개인정보 처리자는 데이터의 관련 분야 및 특수성 등을 고려하여 상황에 따라 유동적으로 처리 가능함

※ 가명정보 처리 가이드라인(개인정보보호위원회(’24.2.)) 01 가이드라인 개요

## 제1장 | 가이드라인 개요 03 3 적용 범위

- (우선순위) 보건의료 분야의 개인정보 가명처리 및 가명정보 처리에 관하여 동 가이드라인 직용

※ 본 가이드라인에서 별도로 정하지 않은 개인정보 보호 법령 및 고시에 규정된 사항은 ｢가명정보 처리 가이드라인｣ (개인정보보호위원회)을 준용해야 함

- - 또한 ｢가명정보 처리 가이드라인｣ (개인정보보호위원회)에서 구체적으로 제시하고 있는 가명처리 및 가명정보 처리에

- 관한 예시나 서식 등에 대해서는 

### 왜 평평한 텍스트보다 markdown 구조화가 유리한가?

markdown으로 구조를 살려두면, 나중에 `헤더 단위 분할`, `섹션별 요약`, `질문과 관련된 부분만 검색`이 쉬워집니다. 반대로 평평한 텍스트는 문서 전체가 한 덩어리처럼 보이기 때문에, 모델이 어느 문장이 어떤 절과 연결되는지 파악하기가 더 어렵습니다.

## 표 과제 - 일반 텍스트 로더와 표 전용 추출 비교

과제의 두 번째 핵심은 표를 `텍스트 로더`로 읽었을 때와 `pdfplumber`로 읽었을 때 결과가 어떻게 달라지는지 직접 확인하는 것입니다.

In [10]:
table_loader = PyPDFLoader(str(TABLE_PDF))
table_docs = table_loader.load()

print('[PyPDFLoader로 읽은 첫 페이지 일부]\n')
print(table_docs[0].page_content[:1800])

def clean_cell(cell):
    if cell is None:
        return ''
    return ' '.join(str(cell).split())

markdown_tables = []
with pdfplumber.open(TABLE_PDF) as pdf:
    for page_idx, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()
        print(f'page {page_idx} -> table count: {len(tables)}')
        for table_idx, raw_table in enumerate(tables, start=1):
            cleaned = [[clean_cell(cell) for cell in row] for row in raw_table if row]
            if len(cleaned) < 2:
                continue
            header = cleaned[0]
            rows = cleaned[1:]
            df = pd.DataFrame(rows, columns=header)
            md_table = df.to_markdown(index=False)
            markdown_tables.append({
                'page': page_idx,
                'table_index': table_idx,
                'dataframe': df,
                'markdown': md_table,
            })

print('\n[pdfplumber로 추출한 markdown 표 미리보기]\n')
for item in markdown_tables:
    print(f"--- page {item['page']} / table {item['table_index']} ---")
    print(item['markdown'])
    print()

[PyPDFLoader로 읽은 첫 페이지 일부]

제5장 | 보건의료데이터 활용 시나리오 65
가명정보 처리 기초자료 명세서
신청기관 정보
기관명 한국대학교병원
주소 서울특별시 종로구 대학로 100-1
데이터명 병리데이터, 흉부CT데이너, 골밀도 
검사(DEXA) 데이터, 외과병리 보고서 평가목적 유방암 진단 자동화 및 골밀도 감소 
여부 확인을 위한 AI 개발 연구
데이터 수집 한국대학병원에서 유방암으로 진단받고 수술한 여성 환자 500명 진료기록
 * 연구대상자 선정기간 (2012.1.1. ~ 2019.12.31.)
이용방법 기관 내 물리적으로 분리된 분석 공간에서 데이터 연구 이용 
아용기간 2023년 5월 1일 ~ 2025년 4월 30일(2년)
데이터 명세
번호 구분 검토사항
1 데이터 
특징
항목명 데이터  유형 예시 검토사항
병리조직
데이터
이미지* 500장
(500명* 1장)
병리슬라이드 번호 사진 등 
개인식별 사항 가명처리 필요
흉부CT
데이터
영상･이미지
(DICOM)
* 100,000장
(500명 * 100장 * 
2회 촬영)
* 이미지 내 환자이름, 생년
월일, 성별, 환자번호 존재
DICOM 영상･이미지에 포함된 
환자관련정보(환자이름, 생년월일, 
환자성별, 환자번호)는 개인식별 
가능성이 있어 가명처리 필요
page 1 -> table count: 3
page 2 -> table count: 2

[pdfplumber로 추출한 markdown 표 미리보기]

--- page 1 / table 1 ---
| 신청기관 정보   |                                                                                                                      |          |                                                                 |
|:----------------|:---------------

### 비교 결과 정리

- `PyPDFLoader`는 PDF 안 글자를 읽어 오지만, 표의 행/열 관계까지 복원하지는 못합니다.
- `pdfplumber`는 셀 경계와 정렬을 활용해 표 구조를 복원하므로 `어느 값이 어느 열에 속하는지`를 유지할 수 있습니다.
- 그래서 표 데이터는 일반 텍스트 로더보다 표 전용 도구로 추출한 뒤 markdown 표로 바꾸는 편이 이후 검색과 요약에 훨씬 유리합니다.


## 4단계 - 가공한 데이터 활용하기

이제 3단계에서 만든 markdown과 표 markdown을 LangChain에 넣어 검색/질의응답에 사용합니다.

구성은 다음과 같습니다.

- 서술형 markdown 문서를 헤더 기준으로 쪼개기
- 표 markdown도 별도 Document로 추가하기
- `FAISS + OpenAIEmbeddings`로 벡터 검색 만들기
- `ChatOpenAI`로 검색 결과를 근거로 답변 생성하기


In [11]:
def split_markdown_sections(md_text: str):
    sections = []
    current_title = '문서 전체'
    current_lines = []

    for line in md_text.splitlines():
        if line.startswith('#'):
            if current_lines:
                sections.append((current_title, '\n'.join(current_lines).strip()))
            current_title = line.lstrip('#').strip()
            current_lines = [line]
        else:
            current_lines.append(line)

    if current_lines:
        sections.append((current_title, '\n'.join(current_lines).strip()))
    return sections

documents = []
for title, content in split_markdown_sections(markdown_text):
    documents.append(Document(page_content=content, metadata={'source': 'excerpt', 'section': title}))

for item in markdown_tables:
    documents.append(
        Document(
            page_content=item['markdown'],
            metadata={
                'source': 'table',
                'section': f"table_page_{item['page']}_{item['table_index']}"
            },
        )
    )

print('검색용 Document 수:', len(documents))
for doc in documents[:3]:
    print(doc.metadata)

검색용 Document 수: 8
{'source': 'excerpt', 'section': '보건의료데이터 활용 가이드라인 개요'}
{'source': 'excerpt', 'section': '제1장 | 가이드라인 개요 03 3 적용 범위'}
{'source': 'excerpt', 'section': '제1장 | 가이드라인 개요 05 통계작성: 통계란 특정 집단이나 대상 등에 관하여 작성한 수량적인 정보를 의미하여 시장조사와'}


In [13]:
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

sample_questions = [
    '가명처리란 무엇인가요?',
    '이 가이드라인은 누구에게 적용되나요?',
    '가명정보는 어떤 목적으로 활용할 수 있나요?',
]

if not os.getenv('OPENAI_API_KEY'):
    print('OPENAI_API_KEY가 없어 LLM 답변 생성은 건너뜁니다.')
    print('대신 검색 대상 Document 개수와 샘플 질문을 출력합니다.')
    print('검색용 Document 수:', len(documents))
    for question in sample_questions:
        print('-', question)
else:
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    vectorstore = FAISS.from_documents(documents, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

    def ask_rag(question: str):
        retrieved_docs = retriever.invoke(question)
        context = '\n\n'.join(
            f"[source={doc.metadata}]\n{doc.page_content}" for doc in retrieved_docs
        )
        prompt = f'''너는 반드시 제공된 문맥만 근거로 답하는 도우미다.
질문: {question}

문맥:
{context}

요구사항:
- 문맥에 있는 내용만 사용한다.
- 문맥에 없으면 없다고 말한다.
- 답변 뒤에 근거가 된 section 이름을 짧게 덧붙인다.
'''
        answer = llm.invoke(prompt)
        return retrieved_docs, answer.content

    for question in sample_questions:
        docs_found, answer = ask_rag(question)
        print('=' * 100)
        print('질문:', question)
        print('답변:', answer)
        print('검색된 section:', [doc.metadata['section'] for doc in docs_found])
        print()

질문: 가명처리란 무엇인가요?
답변: 가명처리란 개인정보의 일부를 삭제하거나 일부 또는 전부를 대체하는 등의 방법으로, 추가 정보가 없이는 특정 개인을 알아볼 수 없도록 처리하는 것을 의미합니다. 이 과정에서 생성된 정보는 '가명정보'라고 하며, 이는 원래의 상태로 복원하기 위한 추가정보의 사용이나 결합 없이는 특정 개인을 식별할 수 없는 정보입니다. 

(제1장 | 가이드라인 개요 03 3 적용 범위)
검색된 section: ['제1장 | 가이드라인 개요 03 3 적용 범위', '제1장 | 가이드라인 개요 05 통계작성: 통계란 특정 집단이나 대상 등에 관하여 작성한 수량적인 정보를 의미하여 시장조사와', '보건의료데이터 활용 가이드라인 개요']

질문: 이 가이드라인은 누구에게 적용되나요?
답변: 이 가이드라인은 의료기관, 연구자, 기업, 공공기관, 대학교 등 보건의료데이터를 처리하는 모든 개인정보처리자에게 적용됩니다. (제1장 | 가이드라인 개요 03 3 적용 범위)
검색된 section: ['보건의료데이터 활용 가이드라인 개요', '제1장 | 가이드라인 개요 03 3 적용 범위', '제1장 | 가이드라인 개요 05 통계작성: 통계란 특정 집단이나 대상 등에 관하여 작성한 수량적인 정보를 의미하여 시장조사와']

질문: 가명정보는 어떤 목적으로 활용할 수 있나요?
답변: 가명정보는 통계작성, 과학적 연구, 공익적 기록보존 등의 목적으로 활용할 수 있습니다. 이는 개인정보처리자가 개인정보를 가명처리하여 이러한 목적을 위해 활용할 수 있는 법적 근거가 마련되었기 때문입니다. (보건의료데이터 활용 가이드라인 개요)
검색된 section: ['보건의료데이터 활용 가이드라인 개요', '제1장 | 가이드라인 개요 03 3 적용 범위', 'table_page_2_1']



## 제출용 정리

### 3단계 markdown 변환 결과 요약

- PDF를 페이지 단위로 불러온 뒤 본문을 하나로 합쳤다.
- 반복되는 머리말, 페이지 번호, 끊어진 줄바꿈을 정리해 clean text를 만들었다.
- 제목과 항목 구조를 살린 markdown으로 바꿔, 이후 섹션 단위 검색이 가능하도록 만들었다.
- 표는 일반 텍스트가 아니라 `pdfplumber`로 셀 구조를 복원한 뒤 markdown 표로 변환했다.

### 생각해볼 질문 1

**같은 PDF라도 평평한 텍스트로 둘 때와 markdown으로 구조화했을 때, 이후 요약이나 검색에서 어떤 차이가 생길까요?**

평평한 텍스트는 문서 전체가 한 덩어리로 섞여 있어서, 모델이 어느 문장이 제목에 속하고 어느 문장이 세부 설명인지 구분하기 어렵다. 반면 markdown 구조는 제목과 하위 항목의 경계를 분명히 보여주므로, 특정 절만 잘라 검색하거나 섹션별 요약을 만들기가 쉬워진다. 즉 구조화는 단순한 보기 편의가 아니라 검색 정확도와 답변 근거성을 높이는 전처리다.

### 생각해볼 질문 2

**외부 데이터(PDF, 웹페이지, CSV 등)마다 가공 방식이 달라져야 하는 이유는 무엇일까요?**

외부 데이터는 형식마다 구조와 노이즈가 다르기 때문이다. PDF는 줄바꿈, 머리말, 페이지 번호, 표 깨짐 문제가 많고, 웹페이지는 메뉴나 광고 같은 주변 요소 제거가 중요하며, CSV는 열 이름과 결측치 처리가 핵심이다. 즉 데이터마다 원래 구조와 오염 방식이 다르므로, 이후 검색과 요약에 쓰기 좋은 형태로 만들기 위한 가공 전략도 달라져야 한다.

### 표 과제 질문 1

**일반 텍스트 로더로 표를 읽으면 왜 행과 열이 뒤섞일까요?**

PDF 내부에서 표는 진짜 데이터베이스 표처럼 저장되지 않고, 특정 좌표에 배치된 글자들의 묶음으로 존재하는 경우가 많다. 그래서 일반 텍스트 로더는 글자의 읽기 순서만 따라가고, 어느 값이 어느 셀에 속하는지까지는 복원하지 못한다. 그 결과 행과 열 관계가 사라진 평평한 텍스트가 된다.

### 표 과제 질문 2

**표를 markdown으로 잘 정리해두면, 이후 요약이나 검색에서 어떤 이점이 생길까요?**

markdown 표는 항목명과 값의 대응 관계를 유지하므로, 특정 조건에 맞는 행을 찾거나 한 열만 비교하는 작업이 쉬워진다. 그래서 질문 응답에서 정확도가 높아지고, 표 전체를 설명하거나 일부 항목만 추출하는 작업도 더 안정적으로 수행할 수 있다.
